# SysML v2 Python Client - Basic Usage Example

This notebook demonstrates basic usage of the `sysmlv2_client` and SysML SDK library to interact with a running SysML v2 Flexo API server in Kotar or Standalone Flexo (like the local Flexo instance).

## 1. Setup

Make sure:
1. Install Kotar desktop
2. Start Kotar desktop
3. create a .venv
4. pip install sysml-api-client=0.1.0  
5. pip install sysml=0.2.0
6. Copy the bearer token from Kotar desktop
7. The `sysmlv2_client` package is accessible (e.g., by running this notebook from the project root directory or installing the package).

### Install Kotar desktop
![Description](Clipboard10.png)
![Description](Clipboard11.png)
### Open a local folder
![Description](Clipboard12.png)
![Description](Clipboard13.png)

### Open VSCode, create a .venv and install ipykernel,
1. pip install sysml-api-client=0.1.0  
2. pip install sysml=0.2.0

![Description](Clipboard14.png)
![Description](Clipboard23.png)
![Description](Clipboard17.png)
![Description](Clipboard18.png)

### Create SysML model
![Description](Clipboard25.png)

### Copy URL and token into notebook
![Description](Clipboard20.png)


In [16]:
import sys
import os
import uuid # To generate unique IDs if needed by API for creation
sys.path.insert(0, os.path.abspath('../src'))

from sysmlv2_client import SysMLV2Client, SysMLV2Error, SysMLV2NotFoundError
import json 
from pprint import pprint

## 2. Initialize the Client

Replace `"Bearer YOUR_TOKEN_HERE"` with the actual bearer token obtained from Kotar or your Flexo instance.

In [17]:
BASE_URL = "http://127.0.0.1:62859" 
# !!! IMPORTANT: Replace with your actual token !!!
BEARER_TOKEN = "Bearer 0cf8164f4c0395bb749b426826ea4eebe6de9d6258a8b5fbc369834c908255ee"

client = None
try:
    client = SysMLV2Client(base_url=BASE_URL, bearer_token=BEARER_TOKEN)
    print("Client initialized successfully!")
except ValueError as e:
    print(f"Error initializing client: {e}")
except Exception as e:
    print(f"An unexpected error occurred during initialization: {e}")

Client initialized successfully!


## 3. Project Operations

## Get projects

In [18]:
token = BEARER_TOKEN.removeprefix("Bearer ").strip()

client._session.headers.pop("Authorization", None)
client._session.headers["x-sysmlv2-svc-token"] = token
client._session.trust_env = False

if client:
    try:
        print("--- Getting Projects ---")
        projects = client.get_projects()
        print(f"Found {len(projects)} projects.")
        if projects:
            for project in projects:
                proj_id = project.get('@id', 'N/A')
                proj_name = project.get('name', 'N/A')
                print(f"  - Name: {proj_name}, ID: {proj_id}")
        else:
            print("  (No projects found)")
    except SysMLV2Error as e:
        print(f"Error getting projects: {e}")

--- Getting Projects ---
Found 3 projects.
  - Name: kotar, ID: 42b035aa-9c8e-4530-ac9d-c914ce58e70f
  - Name: My Client Example Project, ID: a2931cb4-8100-49c4-ae48-a9ca4c3bc184
  - Name: test01, ID: face4d6f-3cfa-4a2a-8913-0b7e4953c9f5


## Create project

In [19]:
# Create a new project (adjust data as needed)
created_project = None
example_project_id = None # Initialize here to ensure it exists

if client:
    new_project_data = {
        "@type": "Project",
        "name": "test02",
        "description": "A project created via the Python client for testing"
    }
    try:
        print("\n--- Creating Project ---")
        created_project = client.create_project(new_project_data)
        print("Project created successfully:")
        pprint(created_project)
        # Store the ID for later use
        example_project_id = created_project.get('@id')
        if not example_project_id:
             print("\n*** WARNING: Could not extract project ID ('@id') from response! Subsequent steps might fail. ***")
    except SysMLV2Error as e:
        print(f"Error creating project: {e}")
else:
    print("Client not initialized, skipping project creation.")


--- Creating Project ---
Project created successfully:
{'@id': '88554119-2cb9-460e-892f-620326f94d04',
 '@type': 'Project',
 'created': '2026-11-14T19:01:47.704Z',
 'defaultBranch': {'@id': 'd54d9da9-8bc5-4817-b2f6-1b83980fb910'},
 'description': 'A project created via the Python client for testing',
 'name': 'test02'}


## 4. Commit and Element Operations

Create and commit model in Kotar

# 5. Get Elements

## From autosave branch

In [21]:
project_id = "42b035aa-9c8e-4530-ac9d-c914ce58e70f"

branches = client.list_branches(project_id)

for branch in branches:
    head = branch.get("head") or branch.get("referencedCommit") or {}
    print(branch["name"], branch["@id"], head.get("@id"))

branch_name = "autosave:default"
branch = next(
    (b for b in branches if b.get("name") == branch_name),
    None,
)
if branch is None:
    raise ValueError(f"Branch {branch_name!r} not found; choose from the list above")

head = branch.get("head") or branch.get("referencedCommit") or {}
commit_id = head.get("@id")
if not commit_id:
    raise ValueError("Selected branch has no head commit")

elements = client.list_elements(project_id, commit_id)

print(f"\nBranch: {branch_name}, commit: {commit_id}")
print(f"Found {len(elements)} elements")

for elem in elements:
    name = elem.get("declaredName") or elem.get("name")
    print(f"{elem.get('@type', ''):30} {str(name):25} {elem.get('@id')}")

autosave:default 2c82dae8-56f4-455e-b132-f968fcdc530d f09dcaf3-7f28-4293-b08b-5b9bb6a55df7
default 67c8b70c-a0eb-4b21-878a-8679e200755a efecece8-01a2-406a-bf73-706272d9dc92
Initial 706350fc-1bea-4fad-9704-1cdef758bd61 275ab558-bb55-4ce2-a987-3cf549e8049b

Branch: autosave:default, commit: f09dcaf3-7f28-4293-b08b-5b9bb6a55df7
Found 25 elements
OwningMembership               None                      025b79e1-86c8-5ce1-88f8-d104054458cf
PartUsage                      p8                        05504ccc-b2ae-5b2c-b3f6-34f4cbce80c0
FeatureMembership              None                      3abda591-7e11-5fc8-95d1-3891152e4799
OwningMembership               None                      468cd5fd-ab29-5b71-af30-85d1ca72ce0d
PartUsage                      p5                        51ae5763-2830-58b3-b7f3-8022e5739bc6
MembershipImport               None                      5540885a-d329-5ea5-911d-b110affcd4e0
Package                        pkg                       653e0383-d03f-5abe-8f02-22cc631e16

## Promote auto-save branch to default branch


In [22]:
project = client.get_project_by_id(project_id)
default_branch_id = project["defaultBranch"]["@id"]

# elements read from the chosen autosave commit
commit = client.create_commit(
    project_id,
    {
        "@type": "Commit",
        "description": "Publish model from autosave",
        "change": [
            {
                "@type": "DataVersion",
                "identity": {"@id": element["@id"]},
                "payload": element,
            }
            for element in elements
        ],
    },
    branch_id=default_branch_id,
)

>>> SysMLv2_Python_Client::DEBUG create_commit
/projects/42b035aa-9c8e-4530-ac9d-c914ce58e70f/commits?branchId=706350fc-1bea-4fad-9704-1cdef758bd61


## From default branch

In [23]:
from pprint import pprint
from sysmlv2_client import SysMLV2Error

# Equivalent to get_last_commit_from_project() in api_lib.py
def get_last_commit_from_project(client, proj_obj):
    project_id = proj_obj["@id"]
    default_branch_id = proj_obj["defaultBranch"]["@id"]

    branch = client._request(
        method="GET",
        endpoint=f"/projects/{project_id}/branches/{default_branch_id}",
        expected_status=200
    )

    return branch.get("referencedCommit", {}).get("@id")


# --- List elements from latest commit ---

pkg_element_id = None
p1_element_id = None
example_project_id = '42b035aa-9c8e-4530-ac9d-c914ce58e70f'

if client and example_project_id:
    try:
        # Retrieve project object from project list
        projects = client.get_projects()

        project = next(
            (p for p in projects
             if p.get("@id") == example_project_id),
            None
        )

        if project is None:
            raise ValueError(
                f"Project {example_project_id} not found"
            )

        # Get latest commit
        latest_commit_id = get_last_commit_from_project(
            client, project
        )

        print(f"Project: {project.get('name')}")
        print(f"Latest Commit: {latest_commit_id}")

        if not latest_commit_id:
            raise ValueError("No commit found on default branch")

        # Retrieve elements
        elements = client.list_elements(
            example_project_id,
            latest_commit_id
        )

        print(f"\nFound {len(elements)} elements:\n")

        for elem in elements:
            name = elem.get("name")
            elem_type = elem.get("@type")
            elem_id = elem.get("@id")

            print(f"{elem_type:30} {str(name):25} {elem_id}")

            if name == "pkg" and elem_type == "Package":
                pkg_element_id = elem_id

            elif name == "p1" and elem_type == "PartUsage":
                p1_element_id = elem_id

        print("\n--- Identified elements ---")
        print(f"Package pkg ID: {pkg_element_id}")
        print(f"Part p1 ID:     {p1_element_id}")

        print("\n--- Full element data ---")
        pprint(elements)

    except (SysMLV2Error, ValueError, KeyError) as e:
        print(f"Error: {e}")

else:
    print("Client or project ID missing.")

Project: kotar
Latest Commit: 22da1e20-126a-4cff-a051-1845a3fdcfa2

Found 25 elements:

OwningMembership               None                      025b79e1-86c8-5ce1-88f8-d104054458cf
PartUsage                      p8                        05504ccc-b2ae-5b2c-b3f6-34f4cbce80c0
FeatureMembership              None                      3abda591-7e11-5fc8-95d1-3891152e4799
OwningMembership               None                      468cd5fd-ab29-5b71-af30-85d1ca72ce0d
PartUsage                      p5                        51ae5763-2830-58b3-b7f3-8022e5739bc6
MembershipImport               None                      5540885a-d329-5ea5-911d-b110affcd4e0
Package                        pkg                       653e0383-d03f-5abe-8f02-22cc631e1677
FeatureValue                   None                      6d2989cf-b60b-53fd-892e-0841d4d77b7c
OwningMembership               None                      73d916b9-5893-513d-8a25-b3a79a48ba1e
LiteralRational                None                      7e244bed-

# Use SysML SDK 0.2.0 to traverse model

In [24]:
from collections import defaultdict

from sysml import Model
from sysml.classes import Package, PartDefinition, PartUsage

# Reuse your authenticated SysMLV2Client named "client".
PROJECT_ID = "42b035aa-9c8e-4530-ac9d-c914ce58e70f"
BRANCH_NAME = "autosave:default"

# 1. Select a specific branch and capture its current commit.
branches = client.list_branches(PROJECT_ID)
branch = next(
    (b for b in branches if b.get("name") == BRANCH_NAME),
    None,
)
if branch is None:
    raise ValueError(f"Branch {BRANCH_NAME!r} not found")

head = branch.get("head") or branch.get("referencedCommit") or {}
commit_id = head.get("@id")
if not commit_id:
    raise ValueError("Selected branch has no head commit")

# 2. Load the committed JSON into typed SDK objects.
elements = client.list_elements(PROJECT_ID, commit_id)
if not elements:
    raise ValueError("Selected commit contains no elements")

model = Model.from_full_json(elements)
print(f"Loaded commit: {commit_id}")

# 3. Resolve elements by qualified name.
pkg = model.resolve("pkg")
p1 = model.resolve("pkg::p1")

assert isinstance(pkg, Package)
assert isinstance(p1, PartUsage)

print("Package:", pkg.qualifiedName)
print("Part:", p1.qualifiedName)
print("Part owner:", p1.owner.qualifiedName)

# 4. Query all parts using their SysML metaclass.
print("\nAll parts:")
for part in model.elements_of_type(PartUsage):
    print(f"  {part.qualifiedName} — {part.elementId}")

# 5. Build an ownership tree using typed owner references.
children = defaultdict(list)

for element in model.all():
    owner = element.owner
    if owner is not None:
        children[owner.elementId].append(element)

visited = set()

def walk(element, depth=0):
    if element.elementId in visited:
        return
    visited.add(element.elementId)

    print("  " * depth + (
        f"{type(element).__name__}: "
        f"{element.qualifiedName or element.declaredName or '(unnamed)'}"
    ))

    for child in sorted(
        children[element.elementId],
        key=lambda e: e.declaredName or "",
    ):
        # Show packages and parts; omit relationship records.
        if isinstance(child, (Package, PartDefinition, PartUsage)):
            walk(child, depth + 1)

print("\nPackages and parts:")
walk(pkg)

Loaded commit: f09dcaf3-7f28-4293-b08b-5b9bb6a55df7
Package: pkg
Part: pkg::p1
Part owner: pkg

All parts:
  pkg::p8 — 05504ccc-b2ae-5b2c-b3f6-34f4cbce80c0
  pkg::p5 — 51ae5763-2830-58b3-b7f3-8022e5739bc6
  pkg::p6 — 8a8be12c-0363-5b33-b3b9-34a5d4735c0f
  pkg::p7 — 96a1b836-cc3e-59cf-a9a9-6c836da1862c
  pkg::p1 — acc7882e-85d5-5200-82fa-d1f0ba7146eb
  pkg::p2 — b5ae5724-d3d0-588a-a5ce-c33e32b14e47
  pkg::p3 — e8c047c8-107e-5e81-91d0-a1b40a21c1e4

Packages and parts:
Package: pkg
  PartUsage: pkg::p1
  PartUsage: pkg::p2
  PartUsage: pkg::p3
  PartUsage: pkg::p5
  PartUsage: pkg::p6
  PartUsage: pkg::p7
  PartUsage: pkg::p8


## Show feature values

In [26]:
from sysml.classes import (
    AttributeUsage, FeatureValue, Expression,
    LiteralBoolean, LiteralInteger, LiteralRational, LiteralString,
)

def attribute_value_text(attribute):
    relationships = [
        r for r in attribute.ownedRelationship
        if isinstance(r, FeatureValue)
    ]
    if not relationships:
        return "(no declared value)"
    if len(relationships) != 1:
        return "(multiple value relationships)"

    relationship = relationships[0]
    expression = relationship.value

    # Some exports omit this derived property.
    if expression is None:
        expressions = [
            e for e in relationship.ownedRelatedElement
            if isinstance(e, Expression)
        ]
        if len(expressions) != 1:
            return "(value expression missing from payload)"
        expression = expressions[0]

    if isinstance(expression, (
        LiteralBoolean, LiteralInteger, LiteralRational, LiteralString,
    )):
        value = expression.value
        return repr(value) if value is not None else "(literal value missing)"

    return f"({type(expression).__name__}; not evaluated)"


from collections import defaultdict

from sysml import Model
from sysml.classes import Package, PartDefinition, PartUsage

# Reuse your authenticated SysMLV2Client named "client".
PROJECT_ID = "42b035aa-9c8e-4530-ac9d-c914ce58e70f"
BRANCH_NAME = "autosave:default"

# 1. Select a specific branch and capture its current commit.
branches = client.list_branches(PROJECT_ID)
branch = next(
    (b for b in branches if b.get("name") == BRANCH_NAME),
    None,
)
if branch is None:
    raise ValueError(f"Branch {BRANCH_NAME!r} not found")

head = branch.get("head") or branch.get("referencedCommit") or {}
commit_id = head.get("@id")
if not commit_id:
    raise ValueError("Selected branch has no head commit")

# 2. Load the committed JSON into typed SDK objects.
elements = client.list_elements(PROJECT_ID, commit_id)
if not elements:
    raise ValueError("Selected commit contains no elements")

model = Model.from_full_json(elements)
print(f"Loaded commit: {commit_id}")
for attribute in model.elements_of_type(AttributeUsage):
    print(
        f"{attribute.qualifiedName} = "
        f"{attribute_value_text(attribute)}"
    )

Loaded commit: 147a78df-6858-4301-896d-2404bfc5d196
pkg::p7::a = 20.0
